# VeriTriage — Automated Triage Pipeline
## One script → full verification sign-off prediction report

**This is the scripting contribution:**
Given a new chip placement (cell density + macro region maps),
VeriTriage predicts which sign-off checks will fail
and generates a prioritized verification report.

**Industry impact:**
Engineers skip checks predicted to pass → save hours per iteration.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from datetime import datetime

ROOT = Path(r"C:\Users\TUSHAR\2026-27\PROJECTS\VeriTriage")

# Load models
models = {
    'IR Drop':    joblib.load(ROOT / "models/label_ir_xgb.pkl"),
    'DRC':        joblib.load(ROOT / "models/label_drc_xgb.pkl"),
    'Congestion': joblib.load(ROOT / "models/label_cg_xgb.pkl"),
}

FEATURE_COLS = [c for c in 
    pd.read_csv(ROOT / "data/processed/veritriage_features.csv").columns
    if c.startswith('cd_') or c.startswith('mr_')]

def extract_features(arr):
    h, w = arr.shape
    mid_h, mid_w = h//2, w//2
    return {
        'mean':         arr.mean(),
        'std':          arr.std(),
        'max':          arr.max(),
        'min':          arr.min(),
        'p25':          np.percentile(arr, 25),
        'p50':          np.percentile(arr, 50),
        'p75':          np.percentile(arr, 75),
        'p90':          np.percentile(arr, 90),
        'p95':          np.percentile(arr, 95),
        'p99':          np.percentile(arr, 99),
        'q_topleft':    arr[:mid_h, :mid_w].mean(),
        'q_topright':   arr[:mid_h, mid_w:].mean(),
        'q_botleft':    arr[mid_h:, :mid_w].mean(),
        'q_botright':   arr[mid_h:, mid_w:].mean(),
        'hotspot_row':  np.unravel_index(arr.argmax(), arr.shape)[0] / h,
        'hotspot_col':  np.unravel_index(arr.argmax(), arr.shape)[1] / w,
        'skewness':     float(pd.Series(arr.flatten()).skew()),
        'kurtosis':     float(pd.Series(arr.flatten()).kurtosis()),
        'density_high': (arr > arr.mean() + arr.std()).mean(),
        'density_low':  (arr < arr.mean()).mean(),
        'energy':       (arr**2).mean(),
    }

def run_triage(cell_density_map, macro_region_map, chip_name="unknown"):
    """
    VeriTriage: Predict sign-off verification outcomes from placement maps.
    
    Args:
        cell_density_map: 2D numpy array — cell density from placement
        macro_region_map: 2D numpy array — macro block locations
        chip_name: string identifier for the chip
    
    Returns:
        report: dict with predictions and recommendations
    """
    # Extract features
    cd_feats = {f'cd_{k}': v for k, v in extract_features(cell_density_map).items()}
    mr_feats = {f'mr_{k}': v for k, v in extract_features(macro_region_map).items()}
    all_feats = {**cd_feats, **mr_feats}
    
    X = np.array([[all_feats[f] for f in FEATURE_COLS]])
    
    # Run all 3 predictions
    results = {}
    for task, model in models.items():
        prob     = model.predict_proba(X)[0][1]
        pred     = int(prob > 0.5)
        results[task] = {
            'prediction': 'FAIL' if pred else 'PASS',
            'confidence': round(prob if pred else 1-prob, 3),
            'fail_prob':  round(prob, 3),
        }
    
    return results

def print_triage_report(chip_name, results):
    """Print a formatted verification triage report"""
    print("\n" + "="*55)
    print(f"  VERITRIAGE SIGN-OFF REPORT")
    print(f"  Chip: {chip_name}")
    print(f"  Time: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print("="*55)
    
    checks_to_run  = []
    checks_to_skip = []
    
    for task, res in results.items():
        status = res['prediction']
        conf   = res['confidence']
        prob   = res['fail_prob']
        icon   = "❌ FAIL" if status == 'FAIL' else "✅ PASS"
        
        print(f"\n  {task:12s}: {icon}")
        print(f"    Fail probability: {prob*100:.1f}%")
        print(f"    Confidence:       {conf*100:.1f}%")
        
        if status == 'FAIL':
            checks_to_run.append(task)
        else:
            checks_to_skip.append(task)
    
    print("\n" + "-"*55)
    print(f"  RECOMMENDATION:")
    if checks_to_run:
        print(f"  🔴 RUN these checks:  {', '.join(checks_to_run)}")
    if checks_to_skip:
        print(f"  🟢 SKIP these checks: {', '.join(checks_to_skip)}")
    
    skipped = len(checks_to_skip)
    total   = len(results)
    saving  = skipped / total * 100
    print(f"\n  ⚡ Estimated time saved: {saving:.0f}% of sign-off runtime")
    print("="*55)

print("VeriTriage pipeline ready!")
print("Run triage on any chip with: run_triage(cell_density, macro_region)")

VeriTriage pipeline ready!
Run triage on any chip with: run_triage(cell_density, macro_region)


In [2]:
# Run triage on 5 real chips from the dataset
CD_PATH = ROOT / "data/raw/circuitnet/routability/cell_density/cell_density"
MR_PATH = ROOT / "data/raw/circuitnet/routability/macro_region/macro_region"
IR_PATH = ROOT / "data/raw/circuitnet/IR_drop/IR_drop"
DRC_PATH = ROOT / "data/raw/circuitnet/routability/DRC/DRC_all"

test_chips = sorted(list(CD_PATH.iterdir()))[:5]

print("Running VeriTriage on 5 real chip designs...")

all_correct = 0
all_total   = 0

for chip_path in test_chips:
    name = chip_path.name
    
    cd = np.load(CD_PATH / name)
    mr = np.load(MR_PATH / name)
    ir = np.load(IR_PATH / name)
    drc = np.load(DRC_PATH / name)
    
    # Ground truth
    true_ir  = int(ir.max()    > 5.0)
    true_drc = int(drc.max()   > 1500)
    
    # Run triage
    results = run_triage(cd, mr, chip_name=name)
    print_triage_report(name[:30], results)
    
    # Check accuracy
    pred_ir  = 1 if results['IR Drop']['prediction']  == 'FAIL' else 0
    pred_drc = 1 if results['DRC']['prediction']      == 'FAIL' else 0
    
    correct = int(pred_ir == true_ir) + int(pred_drc == true_drc)
    all_correct += correct
    all_total   += 2
    print(f"  Ground truth — IR: {'FAIL' if true_ir else 'PASS'} | "
          f"DRC: {'FAIL' if true_drc else 'PASS'}")

print(f"\n{'='*55}")
print(f"  Demo accuracy: {all_correct}/{all_total} correct predictions")
print(f"{'='*55}")

Running VeriTriage on 5 real chip designs...

  VERITRIAGE SIGN-OFF REPORT
  Chip: 1-RISCY-a-1-c2-u0.7-m1-p1-f0
  Time: 2026-04-02 02:35

  IR Drop     : ✅ PASS
    Fail probability: 11.2%
    Confidence:       88.8%

  DRC         : ✅ PASS
    Fail probability: 0.9%
    Confidence:       99.1%

  Congestion  : ✅ PASS
    Fail probability: 0.2%
    Confidence:       99.8%

-------------------------------------------------------
  RECOMMENDATION:
  🟢 SKIP these checks: IR Drop, DRC, Congestion

  ⚡ Estimated time saved: 100% of sign-off runtime
  Ground truth — IR: PASS | DRC: PASS

  VERITRIAGE SIGN-OFF REPORT
  Chip: 10-RISCY-a-1-c2-u0.7-m2-p2-f0
  Time: 2026-04-02 02:35

  IR Drop     : ✅ PASS
    Fail probability: 37.9%
    Confidence:       62.1%

  DRC         : ✅ PASS
    Fail probability: 2.1%
    Confidence:       97.9%

  Congestion  : ✅ PASS
    Fail probability: 0.2%
    Confidence:       99.8%

-------------------------------------------------------
  RECOMMENDATION:
  🟢 SK